# Bifrost NØX — WAF v2 bake-off (Colab, GPU runtime)

Trains two DistilBERT arms on `dataset_v2.zip` (shape-balanced: clean and attack values share the same
carrier templates, splits are by payload group, two shapes are held out for OOD tests, and one whole payload
SOURCE is held out in `test_holdout.jsonl`) and judges them against **ship gates** (FP budget on clean traffic,
recall, OOD recall, recall on the held-out payload source, an independent hand-written battery).

* **Arm A** – 7-class softmax, identical to v1's head, so it is a drop-in for `ml/waf/predictor.py`.
* **Arm B** – one sigmoid head per attack family, default-allow (`ml/waf/predictor.py` loads it from `waf_v2.json`).

Runtime → *Change runtime type → T4 GPU*, then run all cells. Upload `dataset_v2.zip` when asked - it MUST be a zip
built with `--holdout-source` (it contains `test_holdout.jsonl`); the notebook stops otherwise.
Nothing here uses lab traffic. `scanner` is tiny/synthetic and is excluded from the gates.

In [ ]:
# 1. deps
!pip -q install "transformers>=4.44" datasets accelerate scikit-learn

In [ ]:
# 2. data
import json, zipfile, os
from google.colab import files
up = files.upload()                      # dataset_v2.zip
zipfile.ZipFile(next(iter(up))).extractall("/content/v2")
def load(n): return [json.loads(l) for l in open(f"/content/v2/{n}.jsonl")]
train, val, test, ood = (load(n) for n in ("train", "val", "test", "test_ood"))
assert os.path.exists("/content/v2/test_holdout.jsonl"), (
    "this dataset_v2.zip has no test_holdout.jsonl - rebuild with `python -m dataset_tools.waf.v2.build --holdout-source fuzzdb` "
    "and upload THAT zip (a default-build zip would silently skip the source-independent gates)")
holdout = load("test_holdout")
hold = [r for r in holdout if r["shape"] != "probe"]           # held-out-source attacks + realistic benign bodies
probe_rows = [r for r in holdout if r["shape"] == "probe"]     # the 25 hand-written probes (independent battery)
assert any(r["label"] != "clean" for r in hold), "test_holdout.jsonl has no held-out-source attack rows"
assert len(probe_rows) >= 20, "test_holdout.jsonl is missing the hand-written probes"
LABELS = ["clean", "sqli", "xss", "path_traversal", "command_injection", "ssti", "scanner"]   # == ml/waf/taxonomy.py
ATTACKS = LABELS[1:]
print({k: len(v) for k, v in dict(train=train, val=val, test=test, ood=ood, hold=hold, probes=probe_rows).items()})
print(open("/content/v2/DATASET_CARD.md").read()[:1500])

In [ ]:
# 3. shared setup
import numpy as np, torch
from collections import Counter
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding)
MODEL, MAX_LEN, EPOCHS = "distilbert-base-uncased", 256, 3      # MAX_LEN must match predictor.TRANSFORMER_MAX_LENGTH
tok = AutoTokenizer.from_pretrained(MODEL)
label2id = {l: i for i, l in enumerate(LABELS)}
def tokenize(batch): return tok(batch["text"], truncation=True, max_length=MAX_LEN)
def ds_single(rows):
    return Dataset.from_dict({"text": [r["text"] for r in rows], "label": [label2id[r["label"]] for r in rows]}).map(tokenize, batched=True)
def ds_multi(rows):
    y = [[1.0 if r["label"] == a else 0.0 for a in ATTACKS] for r in rows]       # clean = all zeros
    return Dataset.from_dict({"text": [r["text"] for r in rows], "labels": y}).map(tokenize, batched=True)
cnt = Counter(r["label"] for r in train)
print("class counts:", dict(cnt))

In [ ]:
# 4. ARM A - 7-class softmax (sqrt-damped class weights)
w = torch.tensor([(len(train) / (len(LABELS) * cnt[l])) ** 0.5 for l in LABELS], dtype=torch.float)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=w.to(out.logits.device))
        return (loss, out) if return_outputs else loss
def args(out):
    return TrainingArguments(output_dir=out, per_device_train_batch_size=32, per_device_eval_batch_size=128,
                             learning_rate=3e-5, num_train_epochs=EPOCHS, weight_decay=0.01,
                             eval_strategy="epoch", save_strategy="no", fp16=torch.cuda.is_available(),
                             logging_steps=200, report_to="none")
model_a = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=label2id)
ta = WeightedTrainer(model=model_a, args=args("/content/armA"), train_dataset=ds_single(train),
                     eval_dataset=ds_single(val), processing_class=tok, data_collator=DataCollatorWithPadding(tok))
ta.train()

In [ ]:
# 5. ARM B - per-attack sigmoid heads, default-allow
model_b = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(ATTACKS), problem_type="multi_label_classification",
    id2label=dict(enumerate(ATTACKS)), label2id={a: i for i, a in enumerate(ATTACKS)})
tb = Trainer(model=model_b, args=args("/content/armB"), train_dataset=ds_multi(train), eval_dataset=ds_multi(val),
             processing_class=tok, data_collator=DataCollatorWithPadding(tok))
tb.train()

In [ ]:
# 6. one decision rule per arm -> (flag_score, predicted_label)
def _batches(texts, bs=256):
    for i in range(0, len(texts), bs):
        yield texts[i:i + bs]
@torch.no_grad()
def run_logits(model, texts):
    model.eval(); dev = next(model.parameters()).device; outs = []
    for chunk in _batches(texts):
        enc = tok(chunk, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev)
        outs.append(model(**enc).logits.float().cpu())
    return torch.cat(outs)
def score_a(texts):
    """predictor.py rule: flagged iff argmax class is an attack, score = its softmax prob."""
    p = torch.softmax(run_logits(model_a, texts), -1).numpy()
    top = p.argmax(1)
    flag = np.where(top != 0, p.max(1), 0.0)
    return flag, [LABELS[i] if i != 0 else "clean" for i in top]
def score_b(texts):
    p = torch.sigmoid(run_logits(model_b, texts)).numpy()
    return p.max(1), [ATTACKS[i] for i in p.argmax(1)]
ARMS = {"A_softmax7": score_a, "B_sigmoid6": score_b}

FP_BUDGET = 0.005          # fraction of VALIDATION clean rows allowed to be flagged
def pick_threshold(fn):
    clean = [r["text"] for r in val if r["label"] == "clean"]
    flag, _ = fn(clean)
    return float(np.quantile(flag, 1 - FP_BUDGET))
THR = {k: pick_threshold(f) for k, f in ARMS.items()}
print("thresholds from val clean FP budget:", THR)

In [ ]:
# 7. evaluate both arms on test / OOD / the held-out payload source
import collections
def evaluate(name, rows, fn, thr):
    flag, pred = fn([r["text"] for r in rows]); flagged = flag > thr
    clean = [i for i, r in enumerate(rows) if r["label"] == "clean"]
    att = [i for i, r in enumerate(rows) if r["label"] not in ("clean", "scanner")]
    nan = float("nan")
    fp = float(np.mean([flagged[i] for i in clean])) if clean else nan
    rec = float(np.mean([flagged[i] for i in att])) if att else nan
    exact = float(np.mean([flagged[i] and pred[i] == rows[i]["label"] for i in att])) if att else nan
    by_shape = collections.defaultdict(list)
    for i in clean: by_shape[rows[i]["shape"]].append(flagged[i])
    shape_fp = {s: float(np.mean(v)) for s, v in by_shape.items()}
    per_class = {}
    for lab in ATTACKS:
        idx = [i for i, r in enumerate(rows) if r["label"] == lab]
        if idx: per_class[lab] = round(float(np.mean([flagged[i] for i in idx])), 3)
    return dict(fp=fp, recall=rec, exact=exact, n_clean=len(clean), n_attack=len(att), shape_fp=shape_fp,
                per_class_recall=per_class)
R = {}
for k, fn in ARMS.items():
    R[k] = {"test": evaluate(k, test, fn, THR[k]), "ood": evaluate(k, ood, fn, THR[k]),
            "holdout": evaluate(k, hold, fn, THR[k])}
    for split in ("test", "ood", "holdout"):
        r = R[k][split]
        print(f"{k:11s} {split:7s} clean-FP {r['fp']:.2%} (n={r['n_clean']})  recall {r['recall']:.2%} (n={r['n_attack']}, exact class {r['exact']:.2%})")
        print("            FP by shape:", {s: f"{v:.1%}" for s, v in sorted(r['shape_fp'].items())})
        print("            recall by class:", r["per_class_recall"])

In [ ]:
# 8. independent hand-written battery: the 25 probes in test_holdout.jsonl (shape "probe"; never in any generator)
BENIGN = {r["name"]: r["text"] for r in probe_rows if r["label"] == "clean"}
ATTACK = {r["name"]: r["text"] for r in probe_rows if r["label"] not in ("clean", "scanner")}     # scanner: excluded from gates
assert len(BENIGN) >= 10 and len(ATTACK) >= 5, "probe rows missing from test_holdout.jsonl"
IND = {}
for k, fn in ARMS.items():
    fb, _ = fn(list(BENIGN.values())); fa, _ = fn(list(ATTACK.values()))
    IND[k] = dict(benign_fp=int((fb > THR[k]).sum()), attacks_caught=int((fa > THR[k]).sum()),
                  n_benign=len(BENIGN), n_attack=len(ATTACK))
    wrong = [n for n, f in zip(BENIGN, fb) if f > THR[k]]
    missed = [n for n, f in zip(ATTACK, fa) if f <= THR[k]]
    print(k, IND[k], "| benign flagged:", wrong, "| attacks missed:", missed)

In [ ]:
# 9. ship gates. GENERATED from dataset_tools/waf/v2/gate.py (GATES + gate_checks): do not edit here, edit the
# repo and regenerate, so Colab and `python -m dataset_tools.waf.v2.gate` can never disagree. Decide the budgets
# BEFORE looking at the numbers. The production-path replay (text_service, all models) runs offline in gate.py.
import math
GATES = {
 "test_fp": 0.01,
 "shape_fp": 0.03,
 "recall": 0.9,
 "ood_fp": 0.02,
 "ood_recall": 0.8,
 "ind_benign_fp_frac": 0.2,
 "ind_attacks_missed_max": 1,
 "holdout_recall": 0.8,
 "holdout_benign_fp": 0.02
}

def gate_checks(m, gates):
    """Notebook gates for ONE arm. ``m`` = {"test","ood","holdout": evaluate() dicts, "ind": {...}}.
    Returns [(name, ok, detail)]. Self-contained on purpose (embedded in the Colab notebook)."""
    def num(x):
        return math.nan if isinstance(x, bool) or not isinstance(x, (int, float)) else float(x)

    def field(d, key):
        return num(d.get(key)) if isinstance(d, dict) else math.nan

    def pct(v):
        return "n/a" if v != v else f"{v:.2%}"

    def whole(v):
        return int(v) if math.isfinite(v) else "?"

    def part(key):
        return m.get(key) if isinstance(m, dict) else None

    test, ood, hold, ind = part("test"), part("ood"), part("holdout"), part("ind")
    out = []

    def add(name, ok, detail):
        out.append((name, bool(ok), detail))

    t_fp = field(test, "fp")
    add(f"test clean FP <= {gates['test_fp']:.0%}", t_fp <= gates["test_fp"], pct(t_fp))
    shape_fp = test.get("shape_fp") if isinstance(test, dict) else None
    per_shape = {str(k): num(v) for k, v in shape_fp.items()} if isinstance(shape_fp, dict) else {}
    worst = math.nan
    if per_shape and all(v == v for v in per_shape.values()):
        worst_name = max(per_shape, key=per_shape.get)
        worst = per_shape[worst_name]
    add(f"worst shape clean FP <= {gates['shape_fp']:.0%}", worst <= gates["shape_fp"],
        f"{pct(worst)} ({worst_name})" if worst == worst else "n/a")
    t_rec = field(test, "recall")
    add(f"test recall >= {gates['recall']:.0%}", t_rec >= gates["recall"], pct(t_rec))
    o_fp, o_rec = field(ood, "fp"), field(ood, "recall")
    add(f"OOD clean FP <= {gates['ood_fp']:.0%}", o_fp <= gates["ood_fp"], pct(o_fp))
    add(f"OOD recall >= {gates['ood_recall']:.0%}", o_rec >= gates["ood_recall"], pct(o_rec))

    n_b, n_a = field(ind, "n_benign"), field(ind, "n_attack")
    b_fp, a_ok = field(ind, "benign_fp"), field(ind, "attacks_caught")
    b_lim = math.floor(n_b * gates["ind_benign_fp_frac"] + 1e-9) if math.isfinite(n_b) else math.nan
    a_lim = n_a - gates["ind_attacks_missed_max"]
    add(f"independent benign FP <= {gates['ind_benign_fp_frac']:.0%} of {whole(n_b)} probes ({whole(b_lim)})",
        b_fp <= b_lim, f"{whole(b_fp)} flagged")
    add(f"independent attacks caught >= {whole(a_lim)} of {whole(n_a)}", a_ok >= a_lim, f"{whole(a_ok)} caught")

    h_rec, h_fp = field(hold, "recall"), field(hold, "fp")
    h_na, h_nc = field(hold, "n_attack"), field(hold, "n_clean")
    add(f"holdout-source recall >= {gates['holdout_recall']:.0%}",
        h_rec >= gates["holdout_recall"] and h_na > 0, f"{pct(h_rec)} of {whole(h_na)} attack rows")
    add(f"holdout benign FP <= {gates['holdout_benign_fp']:.0%}",
        h_fp <= gates["holdout_benign_fp"] and h_nc > 0, f"{pct(h_fp)} of {whole(h_nc)} clean rows")
    return out

PASS, CHECKS = {}, {}
for k in ARMS:
    CHECKS[k] = gate_checks(dict(test=R[k]["test"], ood=R[k]["ood"], holdout=R[k]["holdout"], ind=IND[k]), GATES)
    PASS[k] = all(ok for _, ok, _ in CHECKS[k])
    print(f"== {k}: {'NOTEBOOK GATES PASS (run gate.py for the production replay)' if PASS[k] else 'FAILS GATES'}")
    for name, ok, detail in CHECKS[k]:
        print("   ", "PASS" if ok else "FAIL", name, f"[{detail}]")

In [ ]:
# 10. export (Arm A is the drop-in). Also exports B for the record.
import shutil, hashlib
def export(model, name, extra):
    d = f"/content/{name}"; model.save_pretrained(d); tok.save_pretrained(d)
    json.dump(dict(labels=LABELS, attack_labels=ATTACKS, max_len=MAX_LEN, **extra), open(f"{d}/waf_v2.json", "w"), indent=1)
    shutil.make_archive(d, "zip", d)
    h = hashlib.sha256(open(f"{d}.zip", "rb").read()).hexdigest()
    print(name, "sha256", h); return f"{d}.zip"
za = export(model_a, "waf_distilbert_v2", dict(arm="A_softmax7", recommended_threshold=THR["A_softmax7"], gates=PASS["A_softmax7"]))
zb = export(model_b, "waf_sigmoid_v2", dict(arm="B_sigmoid6", recommended_threshold=THR["B_sigmoid6"], gates=PASS["B_sigmoid6"]))
json.dump(dict(R=R, IND=IND, THR=THR, PASS=PASS, CHECKS=CHECKS), open("/content/waf_v2_results.json", "w"), indent=1, default=float)
from google.colab import files
for f in (za, zb, "/content/waf_v2_results.json"): files.download(f)

## After download (full steps: dataset_tools/waf/README.md, "Running the v2 bake-off")
1. Keep the current model as `ml/waf/artifacts/waf_distilbert_v1` (rollback). Unzip the arm you want to judge into its own
   directory (e.g. `ml/waf/artifacts/waf_distilbert_v2` or `waf_sigmoid_v2`) - do NOT overwrite the shipping directory yet.
2. Judge it: `python -m dataset_tools.waf.v2.gate --results waf_v2_results.json --artifact ml/waf/artifacts/<dir>`
   (recomputes every gate from the raw numbers and replays the probes through `text_service`; non-zero exit = do not ship).
3. Only if it passes: point `WAF_MODEL_PATH` at the directory. Arm A: also set `WAF_ATTACK_THRESHOLD` (what `text_service`
   reads) to `recommended_threshold`. Arm B: the threshold comes from `waf_v2.json`; `WAF_THRESHOLD` overrides it,
   `WAF_ATTACK_THRESHOLD` has no effect.
4. Re-run `tests/test_waf_scope.py tests/test_request_target.py tests/test_text_target.py`, then the VM battery
   (`vm-lab/fw_tests3.ps1`, 30 checks, 0 mismatches). Record the zip's sha256 (printed above) in the README: artifacts are not in git.